In [12]:
import pandas as pd
import numpy as np
import re
import math
from collections import Counter

from numpy.linalg import norm

# NLP Bahasa Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory


In [13]:
df = pd.read_csv("D:\\Tugas Akhir\\sampel\\data\\scholar_articles_rows(1).csv")

df = df[['id', 'title', 'abstract']]
print("Jumlah dokumen:", len(df))
df.head()


Jumlah dokumen: 52


,id,title,abstract
0,1,pembuatan website sekolah menengah pertama neg...,… Praktik ini akan dilakukan pembuatan website...
1,2,Pembuatan Website Sebagai Media Promosi Yang T...,… pembuatan media promosi dan pemasaran berbas...
2,3,Otodidak Web Programming: Membuat Website Edut...,… Tahap Assembly adalah tahap pembuatan semua ...
3,4,Membuat website powerfull menggunakan PHP,… Hal yang perlu diperhatikan dalam pembuatan ...
4,5,Pembuatan Website Katalog Produk UMKM Untuk Pe...,"… perguruan tinggi, kami melakukan kegiatan pe..."


In [14]:
df_sample = df.head(10)
df_sample[['id', 'title', 'abstract']]


,id,title,abstract
0,1,pembuatan website sekolah menengah pertama neg...,… Praktik ini akan dilakukan pembuatan website...
1,2,Pembuatan Website Sebagai Media Promosi Yang T...,… pembuatan media promosi dan pemasaran berbas...
2,3,Otodidak Web Programming: Membuat Website Edut...,… Tahap Assembly adalah tahap pembuatan semua ...
3,4,Membuat website powerfull menggunakan PHP,… Hal yang perlu diperhatikan dalam pembuatan ...
4,5,Pembuatan Website Katalog Produk UMKM Untuk Pe...,"… perguruan tinggi, kami melakukan kegiatan pe..."
5,6,Pembuatan Website Profil Pada Sekolah Menengah...,… yang belum memiliki website dan dalam penyam...
6,7,Kiat jitu membuat website tanpa modal,"… website terhosting gratisan berjenis blog, m..."
7,8,Pembuatan Website Sebagai Sarana Promosi Pariw...,… Website merupakan salah satu sarana dalam me...
8,9,Pembuatan Website sebagai Media Pencitraan dan...,… dimanfaatkan secara efektif adalah pembuatan...
9,10,Pengembangan dan Pembuatan Website: Sebuah Tin...,… website harus melalui berbagai proses terleb...


In [15]:
stop_factory = StopWordRemoverFactory()
stopwords_id = set(stop_factory.get_stop_words())

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()


In [16]:
def is_indonesian(text):
    tokens = text.split()
    indo_count = sum(1 for t in tokens if t in stopwords_id)
    return indo_count > 0


In [17]:
def preprocess_text(text):
    if pd.isna(text):
        return []

    # 1. Text Cleansing
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # 2. Tokenization
    tokens = text.split()

    # 3. Stopword Removal
    tokens = [t for t in tokens if t not in stopwords_id]

    # 4. Stemming (Bahasa Indonesia saja)
    if is_indonesian(text):
        tokens = [stemmer.stem(t) for t in tokens]

    return tokens


In [18]:
df['title_tokens'] = df['title'].apply(preprocess_text)
df['abstract_tokens'] = df['abstract'].apply(preprocess_text)

df[['id', 'title_tokens', 'abstract_tokens']]


,id,title_tokens,abstract_tokens
0,1,"[buat, website, sekolah, tengah, pertama, nege...","[praktik, laku, buat, website, sekolah, smp, n..."
1,2,"[buat, website, media, promosi, percaya]","[buat, media, promosi, pasar, bas, website, tu..."
2,3,"[otodidak, web, programming, membuat, website,...","[tahap, assembly, tahap, buat, semua, objek, b..."
3,4,"[membuat, website, powerfull, menggunakan, php]","[perlu, perhati, buat, web, kreatifitas, bangu..."
4,5,"[buat, website, katalog, produk, umkm, kembang...","[guru, tinggi, laku, giat, buat, website, medi..."
5,6,"[buat, website, profil, sekolah, tengah, perta...","[milik, website, sampai, informasi, perlu, bua..."
6,7,"[kiat, jitu, buat, website, modal]","[website, terhosting, gratis, jenis, blog, mul..."
7,8,"[buat, website, sarana, promosi, pariwisata, s...","[website, rupa, salah, satu, sarana, promosi, ..."
8,9,"[buat, website, media, citra, promosi, desa, k...","[manfaat, efektif, buat, website, desa, websit..."
9,10,"[kembang, buat, website, buah, tinjau, literatur]","[website, lalu, bagai, proses, lebih, jadi, bu..."


In [19]:
all_tokens = []
for tokens in df['title_tokens'].tolist() + df['abstract_tokens'].tolist():
    all_tokens.extend(tokens)

vocab = sorted(set(all_tokens))
len(vocab)


451

In [20]:
def compute_tf(tokens):
    tf = {}
    total = len(tokens)
    counter = Counter(tokens)

    for term, freq in counter.items():
        tf[term] = freq / total

    return tf


In [21]:
df['tf_title'] = df['title_tokens'].apply(compute_tf)
df['tf_abstract'] = df['abstract_tokens'].apply(compute_tf)


In [22]:
N = len(df)
documents = df['title_tokens'].tolist() + df['abstract_tokens'].tolist()

idf = {}
for term in vocab:
    df_t = sum(1 for doc in documents if term in doc)
    idf[term] = math.log10(N / df_t) if df_t > 0 else 0


In [23]:
def compute_tfidf(tf, idf, vocab):
    return np.array([tf.get(term, 0) * idf[term] for term in vocab])


In [24]:
df['tfidf_title'] = df['tf_title'].apply(lambda x: compute_tfidf(x, idf, vocab))
df['tfidf_abstract'] = df['tf_abstract'].apply(lambda x: compute_tfidf(x, idf, vocab))
